In [ ]:

QUICK = True

if QUICK:
    SPECIES = ["human", "chimpanzee", "dog"]
    KS_A, SEEDS_A, EPOCHS_A = (3, 6), 2, 20          # Experiment A: k values, seeds, epochs
    SEEDS_B, SPECIES_B = 1, ["chimpanzee", "dog"]    # Experiment B (human is slower; use QUICK=False)
else:
    SPECIES = ["human", "chimpanzee", "dog"]
    KS_A, SEEDS_A, EPOCHS_A = (3, 4, 5, 6), 10, 20
    SEEDS_B, SPECIES_B = 5, ["human", "chimpanzee", "dog"]
EPOCHS_B = 10
print("QUICK =", QUICK)

In [ ]:
import os
os.makedirs("data", exist_ok=True)
if not all(os.path.exists(f"data/{s}.txt") for s in ["human", "chimpanzee", "dog"]):
    from google.colab import files
    print("Select human.txt, chimpanzee.txt and dog.txt together.")
    for name, content in files.upload().items():
        open(f"data/{name.split(' (')[0]}", "wb").write(content)      # strips Colab's ' (1)' suffix on re-uploads
print(sorted(os.listdir("data")))

In [ ]:


import numpy as np, pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

SPECIES = ["human", "chimpanzee", "dog"]

def load(species, data_dir="data"):
    df = pd.read_table(f"{data_dir}/{species}.txt")
    return df["sequence"].str.upper().tolist(), df["class"].to_numpy()

def kmers(seq, k):
    """Overlapping k-mers (sliding window of step 1), as in the base paper."""
    return [seq[i:i + k].lower() for i in range(len(seq) - k + 1)]

def kmer_sentence(seq, k):
    return " ".join(kmers(seq, k))

def split(species, seed, data_dir="data", test_size=0.2):
    """80:20 stratified split (paper: 80:20). Same seed -> same split for every model."""
    seqs, y = load(species, data_dir)
    idx = np.arange(len(seqs))
    tr, te = train_test_split(idx, test_size=test_size, random_state=seed, stratify=y)
    return [seqs[i] for i in tr], [seqs[i] for i in te], y[tr], y[te]

def kmer_counts(train_seqs, test_seqs, k):
    """CountVectorizer on k-mer 'words'. Vocabulary is learned on the TRAIN set only."""
    cv = CountVectorizer(token_pattern=r"\S+", lowercase=False, dtype=np.float32)
    Xtr = cv.fit_transform([kmer_sentence(s, k) for s in train_seqs])
    Xte = cv.transform([kmer_sentence(s, k) for s in test_seqs])
    return Xtr, Xte

def metrics(y_true, y_pred):
    """Weighted precision/recall/F1 (the paper reports recall == accuracy, i.e. weighted)."""
    p, r, f, _ = precision_recall_fscore_support(y_true, y_pred, average="weighted", zero_division=0)
    return dict(accuracy=accuracy_score(y_true, y_pred), precision=p, recall=r, f1=f)

In [ ]:

import numpy as np
from common import split, kmer_counts, kmer_sentence

def mlp_features(species, k, seed):
    """k-mer counts -> relative frequencies -> standardised (scaler fit on train only)."""
    tr, te, ytr, yte = split(species, seed)
    Ctr, Cte = kmer_counts(tr, te, k)
    Ftr = np.asarray(Ctr.todense()); Fte = np.asarray(Cte.todense())
    Ftr /= np.maximum(Ftr.sum(1, keepdims=True), 1); Fte /= np.maximum(Fte.sum(1, keepdims=True), 1)
    mu, sd = Ftr.mean(0), Ftr.std(0); sd[sd == 0] = 1
    return ((Ftr - mu) / sd).astype(np.float32), ((Fte - mu) / sd).astype(np.float32), ytr, yte

def sparse_features(species, k_word, seed):


    from sklearn.feature_extraction.text import CountVectorizer
    from sklearn.preprocessing import normalize
    tr, te, ytr, yte = split(species, seed)
    cv = CountVectorizer(ngram_range=(4, 4), lowercase=False, token_pattern=r"\S+", dtype=np.float32)
    Ctr = cv.fit_transform([kmer_sentence(s, k_word) for s in tr]); Cte = cv.transform([kmer_sentence(s, k_word) for s in te])
    return Ctr, Cte, ytr, yte, normalize(Ctr.log1p()).astype(np.float32), normalize(Cte.log1p()).astype(np.float32)

In [ ]:



import numpy as np
from scipy.special import erf, expit

_SELU_A, _SELU_S = 1.6732632423543772, 1.0507009873554805
_C = 1 / np.sqrt(2 * np.pi)

def _sp(z):  # softplus, stable
    return np.logaddexp(0, z)

def _relu(z): return np.maximum(0, z), (z > 0).astype(z.dtype)
def _leaky(z): return np.where(z > 0, z, 0.01 * z), np.where(z > 0, 1.0, 0.01).astype(z.dtype)
def _elu(z):
    e = np.exp(np.minimum(z, 0)); return np.where(z > 0, z, e - 1), np.where(z > 0, 1.0, e)
def _selu(z):
    e = np.exp(np.minimum(z, 0))
    return _SELU_S * np.where(z > 0, z, _SELU_A * (e - 1)), _SELU_S * np.where(z > 0, 1.0, _SELU_A * e)
def _gelu(z):
    cdf = 0.5 * (1 + erf(z / np.sqrt(2))); return z * cdf, cdf + z * _C * np.exp(-0.5 * z * z)
def _silu(z):
    s = expit(z); return z * s, s * (1 + z * (1 - s))
def _tanh(z):
    t = np.tanh(z); return t, 1 - t * t
def _sigmoid(z):
    s = expit(z); return s, s * (1 - s)
def _softplus(z): return _sp(z), expit(z)
def _mish(z):
    t = np.tanh(_sp(z)); s = expit(z); return z * t, t + z * s * (1 - t * t)

ACTIVATIONS = {"ReLU": _relu, "LeakyReLU": _leaky, "ELU": _elu, "SELU": _selu, "GELU": _gelu,
               "SiLU": _silu, "Mish": _mish, "Softplus": _softplus, "Tanh": _tanh, "Sigmoid": _sigmoid}
# weight-init scheme matched to each activation (He / LeCun / Xavier)
INIT = {"ReLU": "he", "LeakyReLU": "he", "ELU": "he", "GELU": "he", "SiLU": "he", "Mish": "he",
        "Softplus": "he", "SELU": "lecun", "Tanh": "xavier", "Sigmoid": "xavier"}

def _init(rng, fan_in, fan_out, scheme):
    std = {"he": np.sqrt(2 / fan_in), "lecun": np.sqrt(1 / fan_in),
           "xavier": np.sqrt(2 / (fan_in + fan_out))}[scheme]
    return (rng.standard_normal((fan_in, fan_out)) * std).astype(np.float32)

class MLP:
    def __init__(self, n_in, n_out, activation, hidden=(256, 128), seed=0, l2=1e-4):
        self.f = ACTIVATIONS[activation]; self.rng = np.random.default_rng(seed); self.l2 = l2
        sizes = [n_in, *hidden, n_out]
        self.W = [_init(self.rng, a, b, INIT[activation]) for a, b in zip(sizes[:-1], sizes[1:])]
        self.b = [np.zeros(b, np.float32) for b in sizes[1:]]

    def _forward(self, X, keep=False):
        zs, h = [], X
        for i, (W, b) in enumerate(zip(self.W, self.b)):
            z = h @ W + b
            if i < len(self.W) - 1:
                zs.append((h, z)); h = self.f(z)[0]
            else:
                zs.append((h, z)); z = z - z.max(1, keepdims=True); e = np.exp(z); h = e / e.sum(1, keepdims=True)
        return (h, zs) if keep else h

    def fit(self, X, y, epochs=30, lr=1e-3, batch=64):
        # X may be a dense array or a scipy CSR matrix (used for the high-order k-mer experiment)
        n, C = X.shape[0], self.b[-1].size
        Y = np.eye(C, dtype=np.float32)[y]
        mW = [np.zeros_like(w) for w in self.W]; vW = [np.zeros_like(w) for w in self.W]
        mb = [np.zeros_like(b) for b in self.b]; vb = [np.zeros_like(b) for b in self.b]
        t, b1, b2, eps, losses = 0, 0.9, 0.999, 1e-8, []
        for ep in range(epochs):
            perm = self.rng.permutation(n); tot = 0.0
            for s in range(0, n, batch):
                ix = perm[s:s + batch]; xb, yb = X[ix], Y[ix]
                p, cache = self._forward(xb, keep=True)
                tot += -np.log(p[np.arange(len(ix)), y[ix]] + 1e-12).sum()
                d = (p - yb) / len(ix); gW, gb = [None] * len(self.W), [None] * len(self.W)
                for i in range(len(self.W) - 1, -1, -1):
                    h, z = cache[i]
                    gW[i] = h.T @ d + self.l2 * self.W[i]; gb[i] = d.sum(0)
                    if i > 0:
                        d = (d @ self.W[i].T) * self.f(cache[i - 1][1])[1]
                t += 1
                for i in range(len(self.W)):
                    for P, G, M, V in ((self.W, gW, mW, vW), (self.b, gb, mb, vb)):
                        M[i] = b1 * M[i] + (1 - b1) * G[i]; V[i] = b2 * V[i] + (1 - b2) * G[i] ** 2
                        P[i] -= lr * (M[i] / (1 - b1 ** t)) / (np.sqrt(V[i] / (1 - b2 ** t)) + eps)
            losses.append(tot / n)
        return losses

    def predict(self, X): return self._forward(X).argmax(1)

In [ ]:
import numpy as np, pandas as pd, time, matplotlib.pyplot as plt
from scipy import stats
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import GridSearchCV
from common import split, kmer_counts, metrics
from features import mlp_features, sparse_features
from mlp import MLP, ACTIVATIONS
ACTS = list(ACTIVATIONS); plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
print(ACTS)

In [ ]:
z = np.linspace(-4, 4, 41) + 0.013
for n, f in ACTIVATIONS.items():
    h = 1e-6; num = (f(z + h)[0] - f(z - h)[0]) / (2 * h)
    print(f"{n:10s}", "derivative OK" if np.abs(num - f(z)[1]).max() < 1e-5 else "MISMATCH")
fig, ax = plt.subplots(figsize=(7, 4))
x = np.linspace(-4, 4, 400)
for n, f in ACTIVATIONS.items(): ax.plot(x, f(x)[0], label=n)
ax.set_ylim(-2, 4); ax.legend(ncol=2, fontsize=8, frameon=False); ax.set_title("The ten hidden activations"); plt.show()

In [ ]:
paper = {3: 62.2, 4: 88.4, 5: 95.5, 6: 98.4}   # base paper, human data, Table 6
rows = []
for k in (3, 4, 5, 6):
    tr, te, ytr, yte = split("human", 0)
    Xtr, Xte = kmer_counts(tr, te, k)                                     # plain k-mer counts
    plain = (MultinomialNB(alpha=0.1).fit(Xtr, ytr).predict(Xte) == yte).mean() * 100
    Ctr, Cte, ytr, yte, _, _ = sparse_features("human", k, 0)             # paper-faithful (4-gram over words)
    faithful = (MultinomialNB(alpha=0.1).fit(Ctr, ytr).predict(Cte) == yte).mean() * 100
    rows.append(dict(paper_k=k, effective_kmer=k + 3, paper_reported=paper[k],
                     plain_kmer_NB=round(plain, 1), paper_pipeline_NB=round(faithful, 1)))
pd.DataFrame(rows)

In [ ]:
resA, nbA = [], []
t0 = time.time()
for sp in SPECIES:
    for k in KS_A:
        for seed in range(SEEDS_A):
            Xtr, Xte, ytr, yte = mlp_features(sp, k, seed)
            tr, te, _, _ = split(sp, seed); Ctr, Cte = kmer_counts(tr, te, k)
            gs = GridSearchCV(MultinomialNB(), {"alpha": [0.001, 0.01, 0.1, 0.5, 1.0]}, cv=5).fit(Ctr, ytr)
            nbA.append(dict(species=sp, k=k, seed=seed, activation="Naive Bayes", **metrics(yte, gs.predict(Cte))))
            for act in ACTS:
                net = MLP(Xtr.shape[1], 7, act, seed=seed); net.fit(Xtr, ytr, epochs=EPOCHS_A)
                resA.append(dict(species=sp, k=k, seed=seed, activation=act, **metrics(yte, net.predict(Xte))))
        print(f"{sp} k in {KS_A} done ({time.time() - t0:.0f}s)")
resA, nbA = pd.DataFrame(resA), pd.DataFrame(nbA)

In [ ]:
tabA = pd.concat([resA, nbA]).groupby(["species", "k", "activation"]).accuracy.agg(["mean", "std"]).mul(100).round(1).reset_index()
for sp in SPECIES:
    print(f"\n=== {sp}: test accuracy (%), mean over {SEEDS_A} seeds ===")
    print(tabA[tabA.species == sp].pivot(index="activation", columns="k", values="mean").loc[ACTS + ["Naive Bayes"]])

In [ ]:
fig, axs = plt.subplots(1, len(SPECIES), figsize=(4.3 * len(SPECIES), 3.8))
for ax, sp in zip(np.atleast_1d(axs), SPECIES):
    for i, a in enumerate(ACTS):
        s = tabA[(tabA.species == sp) & (tabA.activation == a)]
        ax.plot(s.k, s["mean"], marker="o", ms=3, color=plt.cm.tab10(i), label=a)
    s = tabA[(tabA.species == sp) & (tabA.activation == "Naive Bayes")]; ax.plot(s.k, s["mean"], "k--", label="Naive Bayes")
    ax.set_title(sp.capitalize()); ax.set_xlabel("k"); ax.set_ylabel("Test accuracy (%)"); ax.set_xticks(list(KS_A))
np.atleast_1d(axs)[-1].legend(fontsize=6, frameon=False, bbox_to_anchor=(1, 1)); plt.tight_layout(); plt.show()

In [ ]:
def paired_vs_relu(df, key):
    out = []
    for g, d in df.groupby(key):
        piv = d.pivot(index="seed", columns="activation", values="accuracy"); ps = {}
        for a in ACTS[1:]:
            diff = piv[a] - piv["ReLU"]
            ps[a] = (diff.mean(), stats.ttest_rel(piv[a], piv["ReLU"]).pvalue if diff.std() > 0 else 1.0)
        order = sorted(ps, key=lambda a: ps[a][1]); m = len(order); prev = 0
        for i, a in enumerate(order):
            adj = min(1, max(prev, ps[a][1] * (m - i))); prev = adj
            out.append(dict(zip(key, g if isinstance(g, tuple) else (g,)), activation=a, diff_pts=round(ps[a][0] * 100, 2), p_holm=round(adj, 4)))
    return pd.DataFrame(out)
if SEEDS_A >= 3:
    pv = paired_vs_relu(resA, ["species", "k"]); display(pv[pv.p_holm < 0.05])

In [ ]:
resB, nbB = [], []
t0 = time.time()
for sp in SPECIES_B:
    for seed in range(SEEDS_B):
        Ctr, Cte, ytr, yte, Xtr, Xte = sparse_features(sp, 5, seed)          # paper k=5 -> 8-mers
        gs = GridSearchCV(MultinomialNB(), {"alpha": [0.001, 0.01, 0.1, 0.5, 1.0]}, cv=3).fit(Ctr, ytr)
        nbB.append(dict(species=sp, seed=seed, activation="Naive Bayes", **metrics(yte, gs.predict(Cte))))
        for act in ACTS:
            net = MLP(Xtr.shape[1], 7, act, hidden=(128, 64), seed=seed); loss = net.fit(Xtr, ytr, epochs=EPOCHS_B, batch=128)
            resB.append(dict(species=sp, seed=seed, activation=act, train_loss=loss[-1], **metrics(yte, net.predict(Xte))))
        print(f"{sp} seed {seed} done ({time.time() - t0:.0f}s)")
resB, nbB = pd.DataFrame(resB), pd.DataFrame(nbB)
tabB = pd.concat([resB, nbB]).groupby(["species", "activation"]).accuracy.mean().mul(100).round(1).unstack(0)
tabB.loc[ACTS + ["Naive Bayes"], SPECIES_B]

In [ ]:
fig, axs = plt.subplots(1, len(SPECIES_B), figsize=(4.3 * len(SPECIES_B), 3.6))
for ax, sp in zip(np.atleast_1d(axs), SPECIES_B):
    v = tabB.loc[ACTS, sp]; ax.bar(ACTS, v, color="#2a6f97"); ax.axhline(tabB.loc["Naive Bayes", sp], color="k", ls="--", lw=1, label="Naive Bayes")
    ax.set_title(f"{sp.capitalize()} (8-mers)"); ax.tick_params(axis="x", rotation=60); ax.set_ylabel("Test accuracy (%)")
np.atleast_1d(axs)[0].legend(frameon=False, fontsize=8); plt.tight_layout(); plt.show()
print("Mean final training loss: Sigmoid", resB[resB.activation == "Sigmoid"].train_loss.mean().round(2), "| ReLU", resB[resB.activation == "ReLU"].train_loss.mean().round(3))

In [ ]:
import io, base64
from IPython.display import Image, display
def show(csv, title):
    print(title); display(pd.read_csv(io.StringIO(csv)))
show('species,k,nb_mean,nb_std,relu_mean,best_act,best_mean,best_std\nhuman,3,0.294,0.012,0.814,ReLU,0.814,0.015\nhuman,4,0.34,0.011,0.862,ReLU,0.862,0.01\nhuman,5,0.445,0.016,0.898,ReLU,0.898,0.011\nhuman,6,0.622,0.015,0.931,SiLU,0.932,0.01\nchimpanzee,3,0.356,0.026,0.782,GELU,0.797,0.023\nchimpanzee,4,0.392,0.023,0.851,GELU,0.855,0.02\nchimpanzee,5,0.465,0.02,0.884,Mish,0.886,0.017\nchimpanzee,6,0.609,0.026,0.899,Sigmoid,0.909,0.014\ndog,3,0.384,0.035,0.674,GELU,0.679,0.034\ndog,4,0.413,0.043,0.712,ELU,0.719,0.036\ndog,5,0.463,0.036,0.741,SiLU,0.755,0.034\ndog,6,0.554,0.049,0.741,Sigmoid,0.777,0.033\n', "Naive Bayes vs best MLP, true k-mers (Experiment A, 10 seeds; accuracies as fractions)")
show('species,paper_k,effective_k,n_features,acc_mean,acc_std,paper_reported\nchimpanzee,3,6,4106.1,0.609,0.026,62.9\nchimpanzee,4,7,16396.4,0.846,0.02,86.9\nchimpanzee,5,8,64737.6,0.909,0.018,91.1\nchimpanzee,6,9,226042.4,0.918,0.016,91.4\ndog,3,6,4163.0,0.557,0.048,51.2\ndog,4,7,16463.4,0.693,0.031,65.2\ndog,5,8,64566.8,0.727,0.028,64.0\ndog,6,9,220268.7,0.731,0.032,69.5\nhuman,3,6,4411.2,0.622,0.015,62.2\nhuman,4,7,16753.1,0.877,0.014,88.4\nhuman,5,8,65272.5,0.953,0.011,95.5\nhuman,6,9,230169.2,0.976,0.007,98.4\n', "Paper reproduction: Naive Bayes with the paper's pipeline vs the paper's reported accuracy (percent)")
show('species,seeds,nb_effk8,nb_std,best_act,best_mean,best_std,relu_mean,nb_effk9_paper_k6,relu_minus_nb,p_relu_vs_nb\nhuman,5,0.949,0.014,LeakyReLU,0.964,0.006,0.963,0.976,0.014,0.086\nchimpanzee,5,0.907,0.026,ELU,0.929,0.015,0.928,0.918,0.02,0.064\ndog,5,0.721,0.029,Tanh,0.746,0.04,0.72,0.735,-0.001,0.883\n', "Experiment B: Naive Bayes vs MLP at 8-mers (5 seeds)")
show('activation,mean_rank,mean_acc,wins,top3\nMish,3.583,0.817,1,4\nSiLU,3.625,0.816,2,5\nGELU,3.75,0.818,3,8\nReLU,4.0,0.816,3,6\nLeakyReLU,4.333,0.816,0,4\nELU,6.083,0.805,0,1\nSoftplus,6.667,0.786,0,2\nTanh,7.167,0.789,0,2\nSigmoid,7.417,0.751,2,3\nSELU,8.375,0.786,0,0\n', "Activation ranking over the 12 settings of Experiment A (mean rank, 1 = best)")

In [ ]:
print('Accuracy by activation at k=6 (full study)')
display(Image(data=base64.b64decode('...')))

In [ ]:
print('Accuracy against k for every activation (full study)')
display(Image(data=base64.b64decode('...')))

In [ ]:
print('Accuracy by activation on 8-mer features (full study)')
display(Image(data=base64.b64decode('...')))